In [2]:
import sys
!{sys.executable} -m pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.4 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [4]:

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [5]:
knowledge_base = [
    "Addis Ababa is the capital of Ethiopia and one of the fastest growing cities in Africa.",
    "Ethiopia is known as the cradle of humanity and has the oldest continuous civilization.",
    "Injera is the national dish of Ethiopia made from teff flour.",
    "The Ethiopian calendar is 7-8 years behind the Gregorian calendar.",
    "Ethiopia has 9 UNESCO World Heritage Sites.",
    "AI and technology startups are rapidly growing in Addis Ababa."
]

# Create embeddings
embeddings = embedder.encode(knowledge_base)
print("Embeddings shape:", embeddings.shape)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("FAISS vector database created!")

Embeddings shape: (6, 384)
FAISS vector database created!


In [6]:
# 3. Retrieval Function
def retrieve(query, top_k=2):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = [knowledge_base[i] for i in indices[0]]
    return results

# Test
print(retrieve("What is the capital of Ethiopia?"))

['Addis Ababa is the capital of Ethiopia and one of the fastest growing cities in Africa.', 'Ethiopia is known as the cradle of humanity and has the oldest continuous civilization.']


In [7]:
# 4. RAG Function
def rag_assistant(query):
    # Step 1: Retrieve relevant information
    context = retrieve(query)
    context_text = "\n".join(context)

    # Step 2: Create prompt with context
    system_prompt = f"""You are a helpful Ethiopian AI assistant. Use the following context to answer accurately.

Context:
{context_text}"""

    # Use the generate function from previous day (or copy the one from Day 10)
    full_prompt = f"""<|system|>
{system_prompt}</s>
<|user|>
{query}</s>
<|assistant|>"""

    # (Add your model generation code here - reuse from Day 10)
    print("Retrieved Context:", context_text)
    print("\nAnswer based on RAG coming soon...")
    return context_text

# Test RAG
print(rag_assistant("Tell me about the capital of Ethiopia and its food."))

Retrieved Context: Addis Ababa is the capital of Ethiopia and one of the fastest growing cities in Africa.
Injera is the national dish of Ethiopia made from teff flour.

Answer based on RAG coming soon...
Addis Ababa is the capital of Ethiopia and one of the fastest growing cities in Africa.
Injera is the national dish of Ethiopia made from teff flour.
